# Soil Fertility Classification Using Machine Learning
**PBL Project | Jupyter Notebook**

This notebook follows the workflow: Data Understanding → Cleaning → EDA → Visualization → Correlation → Feature Selection → Model Training → Evaluation → Interpretation → Prediction.

> **Dataset limitation:** This is source-calibrated synthetic teaching data (1,000 rows), not 1,000 authentic laboratory observations. `Fertility_Class` is a derived label based on a score, not an original measured label. Treat results as educational, not agricultural advice.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

## 2. Load Dataset
Place the CSV file in the same folder as this notebook. Update the path if needed.

In [ ]:
df = pd.read_csv("soil_fertility_classification_1000.csv")
df.head()

## 3. Understand the Dataset

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\\nColumn names:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Fertility class counts:")
display(df["Fertility_Class"].value_counts())

print("\\nClass percentages:")
display((df["Fertility_Class"].value_counts(normalize=True) * 100).round(2))

## 4. Data Cleaning

In [ ]:
# Check and remove exact duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Shape after duplicate removal:", df.shape)

# Review data types
display(df.dtypes)

In [ ]:
# Check categorical values for consistency (spaces and capitalization are not automatically altered)
categorical_cols = df.select_dtypes(include=["object", "category"]).columns
for col in categorical_cols:
    print(f"\\n{col}:")
    print(df[col].value_counts(dropna=False).head(20))

## 5. Missing Value Handling
Numerical columns are imputed with their median; categorical columns with their mode. If a column has no missing values, this operation leaves it unchanged.

In [ ]:
missing_before = df.isna().sum()
display(missing_before[missing_before > 0])

numeric_cols_all = df.select_dtypes(include=np.number).columns
categorical_cols_all = df.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols_all:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols_all:
    if df[col].isna().any() and not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values remaining:", int(df.isna().sum().sum()))
display(df.isna().sum())

## 6. Exploratory Data Analysis (EDA)

In [ ]:
# Target class distribution
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Fertility_Class", order=["Low Fertility", "Medium Fertility", "High Fertility"])
plt.title("Distribution of Soil Fertility Classes")
plt.xlabel("Fertility Class")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of pH and Organic Carbon
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["pH"], bins=20, kde=True, ax=axes[0])
axes[0].set_title("Soil pH Distribution")
axes[0].set_xlabel("pH")
sns.histplot(df["Organic_Carbon_g_kg"], bins=20, kde=True, ax=axes[1])
axes[1].set_title("Organic Carbon Distribution")
axes[1].set_xlabel("Organic Carbon (g/kg)")
plt.tight_layout()
plt.show()

## 7. Visualization of Soil Parameters

In [ ]:
nutrients = ["Available_N_kg_ha", "Available_P_kg_ha", "Available_K_kg_ha"]
for col in nutrients:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x=col, hue="Fertility_Class", bins=20, kde=True)
    plt.title(f"{col} Distribution by Fertility Class")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in ["Available_N_kg_ha", "Organic_Carbon_g_kg", "pH"]:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x="Fertility_Class", y=col, order=["Low Fertility", "Medium Fertility", "High Fertility"])
    plt.title(f"{col} by Fertility Class")
    plt.tight_layout()
    plt.show()

## 8. Correlation Analysis
The fertility score is shown for descriptive inspection only. It will not be used as a model input because it was used to derive the target class.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numerical Columns")
plt.tight_layout()
plt.show()

## 9. Feature Selection
Use soil measurements only. Exclude `Fertility_Score` to reduce direct target leakage because the class label is derived from that score. The 12 numerical soil features below are selected as candidate inputs.

In [ ]:
features = [
    "pH",
    "Electrical_Conductivity_dS_m",
    "Organic_Carbon_g_kg",
    "Available_N_kg_ha",
    "Available_P_kg_ha",
    "Available_K_kg_ha",
    "Sulphur_mg_kg",
    "Zinc_mg_kg",
    "Boron_mg_kg",
    "Iron_mg_kg",
    "Manganese_mg_kg",
    "Copper_mg_kg"
]

X = df[features].copy()
y = df["Fertility_Class"].copy()

print("Candidate input features:", len(features))
print("Target classes:", sorted(y.unique()))

## 10. Train-Test Split
Split the data into 80% training and 20% test sets. Stratification keeps class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

## 11. Feature Scaling and Model Setup
Scaling and feature selection are fitted inside each pipeline on training data only, helping avoid data leakage. Decision Tree and Random Forest do not require scaling, but the common pipeline keeps preprocessing comparable; scaling is especially useful for KNN and SVM.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf")
}

pipelines = {}
for name, model in models.items():
    pipelines[name] = Pipeline([
        ("feature_selection", SelectKBest(score_func=mutual_info_classif, k=8)),
        ("scaler", StandardScaler()),
        ("classifier", model)
    ])

## 12. Train the Four Models

In [ ]:
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    print(f"{name}: training complete")

## 13. Evaluate Models (Accuracy, Precision, Recall, F1)

In [ ]:
results = []
predictions = {}

for name, pipeline in pipelines.items():
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (Macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1 Score (Macro)": f1_score(y_test, y_pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1 Score (Macro)", ascending=False).reset_index(drop=True)
display(results_df.round(4))

In [ ]:
# Metric comparison plot
results_df.set_index("Model")[["Accuracy", "Precision (Macro)", "Recall (Macro)", "F1 Score (Macro)"]].plot(
    kind="bar", figsize=(10, 6)
)
plt.title("Machine Learning Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 14. Confusion Matrices and Classification Reports

In [ ]:
class_names = ["Low Fertility", "Medium Fertility", "High Fertility"]

for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=class_names)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.tight_layout()
    plt.show()

    print(f"\\nClassification Report — {name}")
    print(classification_report(y_test, y_pred, labels=class_names, zero_division=0))

## 15. Select the Best Model
Macro F1-score is used as the main selection metric because it gives equal weight to each class. Accuracy is considered alongside class-wise reports and confusion matrices.

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = pipelines[best_model_name]
best_pred = predictions[best_model_name]

print("Best model by Macro F1:", best_model_name)
print("Accuracy:", round(accuracy_score(y_test, best_pred), 4))
print("Macro Precision:", round(precision_score(y_test, best_pred, average="macro", zero_division=0), 4))
print("Macro Recall:", round(recall_score(y_test, best_pred, average="macro", zero_division=0), 4))
print("Macro F1:", round(f1_score(y_test, best_pred, average="macro", zero_division=0), 4))

## 16. Feature Importance (Random Forest)
This analysis is from the Random Forest fitted inside its pipeline and only uses the features selected by that pipeline. Importance indicates predictive contribution within this model, not causation.

In [ ]:
rf_pipeline = pipelines["Random Forest"]
rf_selector = rf_pipeline.named_steps["feature_selection"]
rf_model = rf_pipeline.named_steps["classifier"]
rf_selected_features = X_train.columns[rf_selector.get_support()]

importance_df = pd.DataFrame({
    "Feature": rf_selected_features,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

## 17. Predict a New Soil Sample
Enter measurements for the 12 candidate features. These example values are illustrative only. Use values with the same units as the dataset.

In [ ]:
new_soil = pd.DataFrame([{
    "pH": 6.8,
    "Electrical_Conductivity_dS_m": 0.45,
    "Organic_Carbon_g_kg": 8.5,
    "Available_N_kg_ha": 250,
    "Available_P_kg_ha": 18,
    "Available_K_kg_ha": 180,
    "Sulphur_mg_kg": 12,
    "Zinc_mg_kg": 0.8,
    "Boron_mg_kg": 0.5,
    "Iron_mg_kg": 8,
    "Manganese_mg_kg": 7,
    "Copper_mg_kg": 0.6
}])[features]

predicted_class = best_model.predict(new_soil)[0]
print("Predicted Soil Fertility Class:", predicted_class)

## 18. Final Results Table

In [ ]:
display(results_df.round(4))
print("Selected final model:", best_model_name)

## 19. Conclusion (Automatically Filled)

In [ ]:
best_row = results_df.iloc[0]

print(f'''
CONCLUSION

This project classified soil fertility into Low Fertility, Medium Fertility,
and High Fertility using a Machine Learning workflow.

Four algorithms were compared: Decision Tree, Random Forest, KNN, and SVM.
The best model by Macro F1-score was {best_row["Model"]}.

Its test metrics were:
Accuracy = {best_row["Accuracy"]:.4f}
Macro Precision = {best_row["Precision (Macro)"]:.4f}
Macro Recall = {best_row["Recall (Macro)"]:.4f}
Macro F1-score = {best_row["F1 Score (Macro)"]:.4f}

Data cleaning, EDA, visualizations, correlation analysis, feature selection,
training, evaluation, and prediction were completed.

Limitation: this is source-calibrated synthetic educational data, and
Fertility_Class is derived from soil parameters. Performance on this dataset
does not prove real-world agricultural accuracy.
''')

## 20. Reproducibility Notes
- Random seed: `42` for the train/test split and models where applicable.
- Test size: 20%, stratified by target class.
- Feature selection: 8 features, performed inside each pipeline.
- Scaling: StandardScaler inside each pipeline.
- Primary comparison metric: Macro F1-score.
- Target leakage prevention: `Fertility_Score` is excluded from input features.

**Reminder:** Do not claim the model is validated on real farms or lab soil samples.